## Training a PPO tiny tacker on the base environment

In [1]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Set directory to tiny_tackers
%cd "/content/drive/My Drive/tiny_tackers"

/content/drive/My Drive/tiny_tackers


In [3]:
# Install and import dependencies
!pip install stable-baselines3 gymnasium pygame

import os
import sys
import time
import pandas as pd
import gymnasium as gym

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.evaluation import evaluate_policy

from gymnasium.wrappers import RecordVideo

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [4]:
# Set path to retrieve for gym environment
BASE_ENV_PATH = "/content/drive/My Drive/tiny_tackers/gym_sailing_environments/gym_sailing_gabo-tor"
sys.path.append(BASE_ENV_PATH)

In [5]:
# Import environment and ensure it's comming from my repo
import gym_sailing
print(gym_sailing.__file__)

/content/drive/My Drive/tiny_tackers/gym_sailing_environments/gym_sailing_gabo-tor/gym_sailing/__init__.py


In [6]:
# Toggle environment action space setups
# ENV_ID = "SailboatDiscrete-v0"
ENV_ID = "Sailboat-v0"
# ENV_ID = "Motorboat-v0"

In [7]:
# Create directory for storing model, logs, videos, and metrics
models_dir = "models/ppo/base"
videos_dir = "videos/ppo/base"
metrics_dir = "metrics"

os.makedirs(models_dir, exist_ok=True)
os.makedirs(videos_dir, exist_ok=True)
os.makedirs(metrics_dir, exist_ok=True)

In [8]:
# Define training and evaluation environments
def make_env(render_mode=None):
    env = gym.make(ENV_ID, render_mode=render_mode)
    env = Monitor(env)
    return env

train_env = make_env()
eval_env = make_env()

/usr/local/lib/python3.12/dist-packages/pygame/pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists
/usr/local/lib/python3.12/dist-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packages
  declare_namespace(pkg)
/usr/local/lib/python3.12/dist-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google.cloud')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-pa

In [9]:
# Improve efficiency with parallel environments
from stable_baselines3.common.env_util import make_vec_env

train_env = make_vec_env(lambda: make_env(), n_envs=4)

In [10]:
# Train PPO
model = PPO(
    policy="MlpPolicy",
    env=train_env,
    learning_rate=3e-4,
    n_steps=1024,
    batch_size=64,
    gamma=0.99,
    verbose=1,
)

start = time.time()

model.learn(
    total_timesteps=1_000_000,
    progress_bar=True,
)

training_time_seconds = time.time() - start

Using cuda device


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Output()

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


Streaming output truncated to the last 5000 lines.
|    loss                 | 3.81        |
|    n_updates            | 170         |
|    policy_gradient_loss | -0.000745   |
|    std                  | 1.01        |
|    value_loss           | 15.1        |
-----------------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 728          |
|    ep_rew_mean          | -236         |
| time/                   |              |
|    fps                  | 853          |
|    iterations           | 19           |
|    time_elapsed         | 91           |
|    total_timesteps      | 77824        |
| train/                  |              |
|    approx_kl            | 0.0018041099 |
|    clip_fraction        | 0.00632      |
|    clip_range           | 0.2          |
|    entropy_loss         | -1.42        |
|    explained_variance   | 0.916        |
|    learning_rate        | 0.0003       |
|    loss

In [11]:
# Evaluate and capture average episode length and reward
episode_rewards, episode_lengths = evaluate_policy(
    model,
    eval_env,
    n_eval_episodes=20,
    deterministic=True,
    return_episode_rewards=True,
)

mean_reward = sum(episode_rewards) / len(episode_rewards)
std_reward = pd.Series(episode_rewards).std()

mean_episode_timesteps = sum(episode_lengths) / len(episode_lengths)
std_episode_timesteps = pd.Series(episode_lengths).std()

print("Mean reward:", mean_reward)
print("Std reward:", std_reward)
print("Mean episode timesteps:", mean_episode_timesteps)
print("Std episode timesteps:", std_episode_timesteps)
print("Training time seconds:", training_time_seconds)

Mean reward: 369.24337660000003
Std reward: 3.8898725163329377
Mean episode timesteps: 1123.55
Std episode timesteps: 40.190958658568185
Training time seconds: 1220.4390835762024


In [13]:
# Save model
model_path = f"{models_dir}/ppo_base_1M"
model.save(model_path)

print(f"Saved model to: {model_path}.zip")

Saved model to: models/ppo/base/ppo_base_1M.zip


In [15]:
# Save metrics
metrics = pd.DataFrame([{
    "env_id": ENV_ID,
    "model": "PPO",
    "training_timesteps": 1_000_000,
    "n_eval_episodes": 20,
    "mean_reward": mean_reward,
    "std_reward": std_reward,
    "mean_episode_timesteps": mean_episode_timesteps,
    "std_episode_timesteps": std_episode_timesteps,
    "training_time_seconds": training_time_seconds,
}])

metrics_path = f"{metrics_dir}/ppo_base_metrics.csv"

if os.path.exists(metrics_path):
    old = pd.read_csv(metrics_path)
    metrics = pd.concat([old, metrics], ignore_index=True)

metrics.to_csv(metrics_path, index=False)
metrics

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,env_id,model,training_timesteps,n_eval_episodes,mean_reward,std_reward,mean_episode_timesteps,std_episode_timesteps,training_time_seconds
0,SailboatDiscrete-v0,PPO,100000,20,-199.105619,58.076113,452.15,452.323973,247.131226
1,Sailboat-v0,PPO,100000,20,369.243377,3.889873,1123.55,40.190959,1220.439084
2,Sailboat-v0,PPO,1000000,20,369.243377,3.889873,1123.55,40.190959,1220.439084


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [16]:
# Record video

video_env = gym.make(ENV_ID, render_mode="rgb_array")
video_env = RecordVideo(
    video_env,
    video_folder=videos_dir,
    name_prefix="ppo_base_trained",
    episode_trigger=lambda episode_id: episode_id == 0,
)

obs, info = video_env.reset()
done = False
truncated = False

while not (done or truncated):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, truncated, info = video_env.step(action)

import builtins # imported to fix bug with recording video
builtins.quit = lambda *args, **kwargs: None # imported to fix bug with recording video
video_env.close()


/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /content/drive/MyDrive/tiny_tackers/videos/ppo/base folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezon